# NpuKit — MNIST tiny-ViT (PYNQ-Z2)

Geometry: native **28×28**, patch **7** → **T=16**, patch vec **49→pad56**, **D=16**, **2 layers** (host-scheduled), **per-stage quant scales**.

Train on the Docker host (torch):
```bash
python3 host/train_vit_mnist.py
```
Then copy `vit_mnist_weights.npz`, `mnist_sample.npz`, `npukit.bit`, and this notebook to the board.

This notebook checks **ref vs board** match and batch accuracy (not 100% classification).

In [1]:
import importlib
import sys

BIT = "/home/xilinx/jupyter_notebooks/npukit.bit"
sys.path.insert(0, "/home/xilinx/jupyter_notebooks")

import npukit_vit_mnist as vit

importlib.reload(vit)
print("T", vit.VIT_T, "D", vit.VIT_D, "L", vit.N_LAYERS)
print("weights", vit.DEFAULT_WEIGHTS, "exists", vit.DEFAULT_WEIGHTS.exists())
print("sample", vit.DEFAULT_SAMPLE, "exists", vit.DEFAULT_SAMPLE.exists())

T 16 D 16 L 2
weights /home/xilinx/jupyter_notebooks/vit_mnist_weights.npz exists True
sample /home/xilinx/jupyter_notebooks/mnist_sample.npz exists True


## Offline ref (trained weights + real MNIST sample)

In [2]:
rc = vit.run_vit_smoke(bit_path=None, seed=0, n=64)
assert rc == 0
print("ref-only return", rc)

ref-only covered by board smoke (same weights/sample)
ref accuracy on this batch: 60/64 (93.8%)
ref-only return 0


## Board: ref vs FPGA + batch accuracy

In [3]:
rc = vit.run_vit_smoke(bit_path=BIT, seed=0, n=64)
assert rc == 0
print("board return", rc)

loaded MNIST sample from /home/xilinx/jupyter_notebooks/mnist_sample.npz (n=64)
=== MNIST tiny-ViT smoke ===
IMG=28 PATCH=7 T=16 D=16 layers=2 patch_dim=49->pad56 classes=10
scales embed act/w=71.86/336.03  L0=40.0/215.8/315.4 L1=12.7/139.3/160.1  cls=15.50/130.40
weights=/home/xilinx/jupyter_notebooks/vit_mnist_weights.npz
Using AXI DMA transport (/home/xilinx/jupyter_notebooks/npukit.bit)
ID=0x4E50554B version=0x00000300 features=0x00000003

--- image[0] label=4 ---
--- ref ---
ref pred=4 logits_q12[:4]=[3629, -8566, 1637, -14609]
--- FPGA ---
hw  pred=4 logits_q12[:4]=[5919, -10307, 1175, -15851]
pred_match: PASS  ref=4 hw=4
tokens: PASS  max|err|=0  tol=512
block0.y_out: PASS  max|err|=113  tol=1024
block1.y_out: info  max|err|=6011
logits: info  max|err|=2359

--- image[1] label=3 ---
--- ref ---
ref pred=3 logits_q12[:4]=[-4991, 6927, 23493, 33832]
--- FPGA ---
hw  pred=3 logits_q12[:4]=[-4813, 4438, 20135, 32813]
pred_match: PASS  ref=3 hw=3
tokens: PASS  max|err|=0  tol=512
blo